# Report - Báo cáo kết quả thực nghiệm `ts2img-lightcnn`

**File này chỉ dùng để đọc và hiển thị kết quả.**

File này **không clone GitHub**, **không đọc code**, và **không dùng thư mục repo**.

Nguồn dữ liệu duy nhất:

```text
/content/drive/MyDrive/ts2img-lightcnn-results
```

Quy trình sử dụng:

1. Chạy `colab_run.ipynb` trước để chạy thực nghiệm.
2. Kết quả được lưu trên Google Drive.
3. Chạy `Report.ipynb` để hiển thị bảng, biểu đồ và file báo cáo phụ trợ.

Tên file đúng là **`Report.ipynb`** với chữ **R viết hoa**.


In [ ]:
# CELL 1 - Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# CELL 2 - Khai báo thư mục kết quả

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# QUAN TRỌNG:
# Report chỉ đọc kết quả từ Google Drive.
# Không clone GitHub trong file này.
RESULTS_ROOT = Path("/content/drive/MyDrive/ts2img-lightcnn-results")

RESULTS_DIR = RESULTS_ROOT / "results"
FIGURES_DIR = RESULTS_ROOT / "figures"
REPORTS_DIR = RESULTS_ROOT / "reports"
OUTPUTS_DIR = RESULTS_ROOT / "outputs"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_ROOT:", RESULTS_ROOT)
print("RESULTS_DIR :", RESULTS_DIR)
print("FIGURES_DIR :", FIGURES_DIR)
print("REPORTS_DIR :", REPORTS_DIR)
print("OUTPUTS_DIR :", OUTPUTS_DIR)

if not RESULTS_ROOT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục {RESULTS_ROOT}. Hãy chạy colab_run.ipynb trước để tạo kết quả."
    )


In [ ]:
# CELL 3 - Kiểm tra các file kết quả hiện có

all_files = sorted([p for p in RESULTS_ROOT.rglob("*") if p.is_file()])

if len(all_files) == 0:
    print("Chưa có file kết quả nào trong thư mục:", RESULTS_ROOT)
    print("Cần chạy colab_run.ipynb trước.")
else:
    print(f"Tìm thấy {len(all_files)} file:")
    for p in all_files:
        print("-", p)


In [ ]:
# CELL 4 - Tìm và đọc file summary_results*.csv

summary_files = sorted(RESULTS_ROOT.rglob("summary_results*.csv"))

if len(summary_files) == 0:
    print("Không tìm thấy file summary_results*.csv.")
    print("\nCác file CSV hiện có:")
    csv_files = sorted(RESULTS_ROOT.rglob("*.csv"))
    for p in csv_files:
        print("-", p)
    raise FileNotFoundError(
        "Chưa có summary_results*.csv. Hãy kiểm tra lại cell chạy thực nghiệm trong colab_run.ipynb."
    )

# Chọn file mới nhất theo thời gian cập nhật.
summary_path = max(summary_files, key=lambda p: p.stat().st_mtime)
print("Đang đọc file:", summary_path)

df = pd.read_csv(summary_path)

print("Kích thước bảng:", df.shape)
display(df)

print("\nDanh sách cột:")
for c in df.columns:
    print("-", c)


In [ ]:
# CELL 5 - Hàm dò tên cột linh hoạt

def find_column(df, candidates):
    lower_map = {str(c).lower().strip(): c for c in df.columns}
    for name in candidates:
        key = name.lower().strip()
        if key in lower_map:
            return lower_map[key]
    return None

dataset_col = find_column(df, ["dataset", "dataset_name", "data", "ucr_dataset"])
method_col = find_column(df, ["method", "model", "representation", "approach", "encoder", "input_type"])
acc_col = find_column(df, ["accuracy", "acc", "test_accuracy", "test_acc", "val_accuracy", "val_acc"])
f1_col = find_column(df, ["macro_f1", "f1_macro", "f1", "test_f1", "macro_f1_score"])
params_col = find_column(df, ["params", "parameters", "num_params", "trainable_params", "n_params"])
train_time_col = find_column(df, ["train_time", "training_time", "train_time_sec", "training_time_sec", "fit_time"])
infer_time_col = find_column(df, ["inference_time", "infer_time", "inference_time_ms", "infer_time_ms", "predict_time"])

print("Cột dataset       :", dataset_col)
print("Cột method/model  :", method_col)
print("Cột accuracy      :", acc_col)
print("Cột macro F1      :", f1_col)
print("Cột params        :", params_col)
print("Cột train time    :", train_time_col)
print("Cột inference time:", infer_time_col)


In [ ]:
# CELL 6 - Tạo bảng rút gọn phục vụ báo cáo

report_columns = []
for c in [dataset_col, method_col, acc_col, f1_col, params_col, train_time_col, infer_time_col]:
    if c is not None and c not in report_columns:
        report_columns.append(c)

if len(report_columns) == 0:
    print("Không dò được cột báo cáo. Hiển thị lại toàn bộ dữ liệu gốc.")
    report_df = df.copy()
else:
    report_df = df[report_columns].copy()

display(report_df)

out_csv = REPORTS_DIR / "report_table_from_summary.csv"
report_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print("Đã lưu bảng rút gọn:", out_csv)


In [ ]:
# CELL 7 - Thống kê mô tả cho các cột số

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) == 0:
    print("Không có cột số để thống kê mô tả.")
else:
    desc_df = df[numeric_cols].describe().T
    display(desc_df)

    out_csv = REPORTS_DIR / "descriptive_statistics.csv"
    desc_df.to_csv(out_csv, encoding="utf-8-sig")
    print("Đã lưu thống kê mô tả:", out_csv)


In [ ]:
# CELL 8 - Tìm phương pháp tốt nhất theo từng bộ dữ liệu

score_col = acc_col if acc_col is not None else f1_col

if dataset_col is None or method_col is None or score_col is None:
    print("Chưa đủ cột dataset/method/score để tìm phương pháp tốt nhất.")
else:
    tmp = df[[dataset_col, method_col, score_col]].copy()
    tmp[score_col] = pd.to_numeric(tmp[score_col], errors="coerce")
    tmp = tmp.dropna(subset=[score_col])

    if len(tmp) == 0:
        print("Không có dữ liệu score hợp lệ.")
    else:
        idx = tmp.groupby(dataset_col)[score_col].idxmax()
        best_df = tmp.loc[idx].sort_values(dataset_col).reset_index(drop=True)
        best_df = best_df.rename(columns={score_col: "best_score"})

        display(best_df)

        out_csv = REPORTS_DIR / "best_method_by_dataset.csv"
        best_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
        print("Đã lưu bảng phương pháp tốt nhất:", out_csv)


In [ ]:
# CELL 9 - Tạo bảng pivot Accuracy nếu có đủ cột

if dataset_col is None or method_col is None or acc_col is None:
    print("Không đủ cột để tạo bảng pivot Accuracy.")
else:
    pivot_acc = df.pivot_table(
        index=dataset_col,
        columns=method_col,
        values=acc_col,
        aggfunc="mean"
    )

    display(pivot_acc)

    out_csv = REPORTS_DIR / "pivot_accuracy.csv"
    pivot_acc.to_csv(out_csv, encoding="utf-8-sig")
    print("Đã lưu bảng pivot Accuracy:", out_csv)


In [ ]:
# CELL 10 - Tạo bảng pivot Macro F1 nếu có đủ cột

if dataset_col is None or method_col is None or f1_col is None:
    print("Không đủ cột để tạo bảng pivot Macro F1.")
else:
    pivot_f1 = df.pivot_table(
        index=dataset_col,
        columns=method_col,
        values=f1_col,
        aggfunc="mean"
    )

    display(pivot_f1)

    out_csv = REPORTS_DIR / "pivot_macro_f1.csv"
    pivot_f1.to_csv(out_csv, encoding="utf-8-sig")
    print("Đã lưu bảng pivot Macro F1:", out_csv)


In [ ]:
# CELL 11 - Vẽ biểu đồ Accuracy theo từng phương pháp

if dataset_col is None or method_col is None or acc_col is None:
    print("Không đủ cột để vẽ biểu đồ Accuracy.")
else:
    plot_df = df[[dataset_col, method_col, acc_col]].copy()
    plot_df[acc_col] = pd.to_numeric(plot_df[acc_col], errors="coerce")
    plot_df = plot_df.dropna(subset=[acc_col])

    if len(plot_df) == 0:
        print("Không có dữ liệu Accuracy hợp lệ.")
    else:
        plt.figure(figsize=(11, 5))
        for dataset in plot_df[dataset_col].dropna().unique():
            sub = plot_df[plot_df[dataset_col] == dataset]
            plt.plot(sub[method_col].astype(str), sub[acc_col], marker="o", label=str(dataset))

        plt.xlabel("Method / Representation")
        plt.ylabel("Accuracy")
        plt.title("Accuracy comparison by method")
        plt.xticks(rotation=45, ha="right")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        out_path = FIGURES_DIR / "accuracy_comparison.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.show()

        print("Đã lưu hình:", out_path)


In [ ]:
# CELL 12 - Vẽ biểu đồ Macro F1 theo từng phương pháp

if dataset_col is None or method_col is None or f1_col is None:
    print("Không đủ cột để vẽ biểu đồ Macro F1.")
else:
    plot_df = df[[dataset_col, method_col, f1_col]].copy()
    plot_df[f1_col] = pd.to_numeric(plot_df[f1_col], errors="coerce")
    plot_df = plot_df.dropna(subset=[f1_col])

    if len(plot_df) == 0:
        print("Không có dữ liệu Macro F1 hợp lệ.")
    else:
        plt.figure(figsize=(11, 5))
        for dataset in plot_df[dataset_col].dropna().unique():
            sub = plot_df[plot_df[dataset_col] == dataset]
            plt.plot(sub[method_col].astype(str), sub[f1_col], marker="o", label=str(dataset))

        plt.xlabel("Method / Representation")
        plt.ylabel("Macro F1-score")
        plt.title("Macro F1-score comparison by method")
        plt.xticks(rotation=45, ha="right")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        out_path = FIGURES_DIR / "macro_f1_comparison.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.show()

        print("Đã lưu hình:", out_path)


In [ ]:
# CELL 13 - Vẽ biểu đồ số tham số nếu có

if method_col is None or params_col is None:
    print("Không đủ cột để vẽ biểu đồ số tham số.")
else:
    param_df = df[[method_col, params_col]].copy()
    param_df[params_col] = pd.to_numeric(param_df[params_col], errors="coerce")
    param_df = param_df.dropna(subset=[params_col])
    param_df = param_df.groupby(method_col, as_index=False)[params_col].mean()

    if len(param_df) == 0:
        print("Không có dữ liệu số tham số hợp lệ.")
    else:
        plt.figure(figsize=(10, 5))
        plt.bar(param_df[method_col].astype(str), param_df[params_col])
        plt.xlabel("Method / Model")
        plt.ylabel("Number of parameters")
        plt.title("Average number of parameters by method")
        plt.xticks(rotation=45, ha="right")
        plt.grid(axis="y")
        plt.tight_layout()

        out_path = FIGURES_DIR / "parameters_by_method.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.show()

        print("Đã lưu hình:", out_path)


In [ ]:
# CELL 14 - Gợi ý nhận xét tự động cho bài báo

print("GỢI Ý NHẬN XÉT:")

if dataset_col is not None:
    print(f"- Số bộ dữ liệu được ghi nhận: {df[dataset_col].nunique()}")

if method_col is not None:
    print(f"- Số phương pháp/mô hình được so sánh: {df[method_col].nunique()}")

if acc_col is not None:
    acc_values = pd.to_numeric(df[acc_col], errors="coerce").dropna()
    if len(acc_values) > 0:
        print(f"- Accuracy trung bình: {acc_values.mean():.4f}")
        print(f"- Accuracy cao nhất: {acc_values.max():.4f}")

if f1_col is not None:
    f1_values = pd.to_numeric(df[f1_col], errors="coerce").dropna()
    if len(f1_values) > 0:
        print(f"- Macro F1 trung bình: {f1_values.mean():.4f}")
        print(f"- Macro F1 cao nhất: {f1_values.max():.4f}")

print("\nĐoạn có thể dùng trong bài báo:")
print(
    "Kết quả thực nghiệm cho thấy hiệu quả phân loại thay đổi theo từng kỹ thuật biểu diễn ảnh 2D "
    "và từng bộ dữ liệu. Do đó, việc lựa chọn GAF, MTF, RP hoặc biểu diễn miền thời gian - tần số "
    "cần căn cứ vào đặc trưng dữ liệu, đồng thời xem xét chi phí mô hình như số tham số, thời gian "
    "huấn luyện và thời gian suy luận."
)


In [ ]:
# CELL 15 - Liệt kê các file báo cáo và hình đã tạo

print("Các file đã tạo trong reports/:")
for p in sorted(REPORTS_DIR.rglob("*")):
    if p.is_file():
        print("-", p)

print("\nCác hình đã tạo trong figures/:")
for p in sorted(FIGURES_DIR.rglob("*")):
    if p.is_file():
        print("-", p)


## Cập nhật file `Report.ipynb` lên GitHub

Sau khi đè file này vào VS Code, chạy:

```bash
git status
git add Report.ipynb
git commit -m "fix Report notebook: read only Drive results"
git push
```

Nếu trong repo vẫn còn file cũ `report.ipynb` chữ thường, đổi tên bằng:

```bash
git mv -f report.ipynb Report.ipynb
git commit -m "rename report notebook to Report"
git push
```
